In [ ]:
import os
from os.path import join
import subprocess
import pandas as pd
import numpy as np
from glob import glob

# Terrain and raster processing libraries (used for DEM-based feature extraction)
import richdem as rd
import rioxarray as rxr
import rasterio as rio
import elevation

import geopandas as gpd
import earthpy as et
from shapely import wkt 

import gc
import matplotlib.pyplot as plt

# Home directory for project repository
home = join(et.io.HOME, "Code", "firebegetsfire_draft")

# Original dataset directory (v1 pipeline outputs / inputs)
data_folder = join(home, "data")

# Second version dataset directory (v2 pipeline outputs and post-processing)
data_v2_folder = join(home, "v2", "reburn_data")

In [ ]:
# Path to filled reburn dataset (v2 processed output used for experimentation)
filled_gdf_path = os.path.join(data_v2_folder, "filled_overlay.gpkg")

# Load processed reburn dataset
filled_test = gpd.read_file(filled_gdf_path)

# Select reburn events where at least one of the two fire periods
# has >= 10% forest cover in the reburn area.
# This is an threshold test
# to explore sensitivity of results to forest cover cutoff values.

true_forested = filled_test[
    (filled_test.reburn1_perc >= 10) |
    (filled_test.reburn2_perc >= 10)
]

true_forested

In [ ]:
# Path to US Census state boundary shapefile (2025 TIGER/Line)
states_path = os.path.join(
    data_v2_folder,
    "tl_2025_us_state",
    "tl_2025_us_state.shp"
)

# Load state boundaries
states = gpd.read_file(states_path)

# Define western US state abbreviations for filtering
states_list = ['WA', 'OR', 'CA', 'ID', 'NV', 'MT', 'WY', 'CO', 'UT', 'AZ', 'NM']

# Subset to western states only
western_states = states[states.STUSPS.isin(states_list)]

# Reproject to match reburn dataset CRS
western_states = western_states.to_crs(filled_test.crs)

# Clip reburn dataset to western US extent, this is done to 
# get rid of outlier events outside of these states' borders
clipped_fill = gpd.clip(filled_test, western_states)

# Output clipped dataset for inspection / downstream exploration
clipped_fill

In [ ]:
# Simple exploratory spatial visualization:
# Western US boundary vs filtered reburn dataset

fig, ax = plt.subplots()

# Plot western state boundaries for geographic context
western_states.boundary.plot(ax=ax)

# Overlay reburn events clipped to western US extent
clipped_fill.plot(ax=ax, color='red')

# Display combined spatial comparison
plt.show()

In [ ]:
# Exploratory forest cover threshold analysis
# These filters test sensitivity of reburn results to different
# minimum forest cover thresholds.

# >= 10% forest cover in either reburn period
forested_10 = clipped_fill[
    (clipped_fill.reburn1_perc >= 10) |
    (clipped_fill.reburn2_perc >= 10)
]

# >= 30% forest cover in either reburn period
forested_30 = clipped_fill[
    (clipped_fill.reburn1_perc >= 30) |
    (clipped_fill.reburn2_perc >= 30)
]

# >= 50% forest cover in either reburn period
forested_50 = clipped_fill[
    (clipped_fill.reburn1_perc >= 50) |
    (clipped_fill.reburn2_perc >= 50)
]

# Output paths for thresholded datasets
forested_10_path = os.path.join(data_v2_folder, "forested_10.gpkg")
forested_30_path = os.path.join(data_v2_folder, "forested_30.gpkg")
forested_50_path = os.path.join(data_v2_folder, "forested_50.gpkg")

# Write each filtered dataset to disk for downstream comparison/analysis
forested_10.to_file(forested_10_path, index=False)
forested_30.to_file(forested_30_path, index=False)
forested_50.to_file(forested_50_path, index=False)

In [ ]:
# Strict forest cover filter
# This version applies a stricter condition than earlier thresholds:
# all three spatial components must exceed 50% forest cover.

forested_50_all = clipped_fill[
    (clipped_fill.reburn1_perc >= 50) &
    (clipped_fill.reburn2_perc >= 50) &
    (clipped_fill.non_reburn_perc >= 50)
]

# Output filtered dataset for inspection
forested_50_all

# Write out filtered dataset
forested_50_all_path = os.path.join(data_v2_folder, "forested_50_all.gpkg")
forested_50_all.to_file(forested_50_all_path, index=False)